# Suite EXE — `txn.exec`: a transaction's suffix over a stream of frames, read-only or writing, through the loopback

Within a transaction, `store.transaction((txn: any = null) => { ... })`, the transaction open, `txn`, executes the
suffix of a transaction's expression over input frames with `txn.exec(frame, remainder, allow_write)`, giving a
stream of frames, an observable in TypeScript, the framework's own, and opening and committing no transaction of its
own ([CAP-308: stacking by mutation transactions, in one round trip](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-308); [the stacked stores proposal, a cache layer and the layer below](https://github.com/pitaman71/mbse-basis/blob/main/proposals/STACKED-STORES.md#a-cache-layer-and-the-layer-below),
its first stage; the first piece of [the transaction-object proposal](https://github.com/pitaman71/mbse-schemas/blob/main/docs/proposals/transaction-object/README.md)). The scope's function names the
transaction with a default, so that it declares no parameter, as an imperative scope's function does. Writing,
`exec` interprets the expression fully, as `store.transaction` does, which collects its stream over one empty input
frame. Read-only, it interprets over what the store holds: where it holds nothing to bind a symbol, or a term would
write, the frame halts and is given carrying its `remainder`, a position in the original expression, which the same
transaction completes, writing: the loopback, a store with no layer below. A partition keeps its join, the rest of a
branch a frame halted in executed by the loopback. Each frame carries the id of the input frame it descends from,
`scope`, and each input frame's frames end with its completion marker, `{"end": scope}`; errors are annotated on
frames ([CAP-309: errors annotated on frames](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-309)), and frames hold references, not visitors ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md#decided)). Everything
commits, or is dropped, with the scope.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import { Builders as B, Plain, Transactions } from "@mbse/schemas/Framework";
import { LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises } from "./support.js";

const { Frame, Remainder } = Transactions;

/** crm.Person, owned by the singleton crm.TheBook under a key, who knows persons; crm.Note, counted. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Knows: S.OfRelation, Person: S.OfObject, Holds: S.OfRelation, Book: S.OfObject, Note: S.OfObject })
    .spec([
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr, age: S.OfInt }).adjacencies([
        (t: any) => t.name("knows").relation([_.Knows]).me("knower"),
        (t: any) => t.relation([_.Knows]).me("known"),
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true),
      ]),
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")]).properties({
        key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      (_: any) => _.Note.name("crm.Note").ref(true).properties({ text: S.OfStr }),
    ])]))]);
}

/** A store holding Ada, 36, who knows Bob, and Bob, of no age: crm.TheBook's contacts "ada" and "bob". */
function made(): any {
  const store: any = new B.Store();
  store.transaction(crm);
  const [Person, Book] = [store.registered("crm.Person"), store.registered("crm.Book")];
  store.transaction((t: any) => t.uid("people/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ a: Person, b: Person, book: Book }).spec([
      (_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.a), bob: (e: any) => e.contact(_.b) }),
      (_: any) => _.a.name("Ada").age(36n).knows([_.b]),
      (_: any) => _.b.name("Bob"),
    ])]))]));
  return store;
}

/** A stream's items, collected as they are given; its error thrown. */
function items(stream: any): any[] {
  const given: any[] = [];
  let failure: { error: unknown } | null = null;
  stream.subscribe({ next: (item: unknown) => given.push(item), error: (error: unknown) => { failure = { error }; } });
  if (failure !== null) throw (failure as { error: unknown }).error;
  return given;
}

/** The frames of a stream's items, its completion markers left out. */
function frames(given: any[]): any[] {
  return given.filter((item) => item instanceof Frame);
}

/** Each item as its scope, a frame's, or the completion marker as plain data. */
function scopes(given: any[]): unknown[] {
  return given.map((item) => item instanceof Frame ? item.scope : item);
}

/** The person a reference names, a visitor of the transaction open, by its key in crm.TheBook's contacts; null for the
 * book itself. */
function designated(store: any, reference: any): any {
  const path = Plain.ToPlain(store)(reference.as_schema(), reference).get("path").get("next");
  return path.has("entry")
    ? store.singleton("crm.TheBook").contacts(Object.fromEntries(path.get("entry").get("properties"))) : null;
}

/** The names of the persons each frame binds, by symbol, the book as "book", read within the transaction open by their
 * references. */
function readNames(store: any, given: any[]): any[] {
  return given.map((frame) => new Map([...frame].map(([symbol, target]: [string, any]) => {
    const found = designated(store, target);
    return [symbol, found === null ? "book" : found.name.get()];
  })));
}

/** `readNames`, within a transaction of its own. */
function names(store: any, given: any[]): any[] {
  return store.transaction(() => readNames(store, given));
}

/** Each person's name, age and whom they know, read within the transaction open. */
function readPeople(store: any): unknown[] {
  return [...store.extent("crm.Person")].map((p: any) => [p.name.get(), store.member(p, "age"),
    [...p.knows].map((e: any) => e.known.name.get()).sort()]).sort((a: any, b: any) => a[0] < b[0] ? -1 : 1);
}

/** `readPeople`, within a transaction of its own. */
function people(store: any): unknown[] {
  return store.transaction(() => readPeople(store));
}

/** The items of `txn.exec`'s stream, within a transaction of their own, which commits. */
function run(store: any, frame: unknown, remainder: unknown, allowWrite: boolean): any[] {
  return store.transaction((txn: any = null) => items(txn.exec(frame, remainder, allowWrite)));
}

/** The error `block` throws. */
function caught(block: () => unknown): any {
  try {
    block();
  } catch (error) {
    return error;
  }
  throw new Error("AssertionError: expected an error");
}

/** A transaction: a map binding Ada to x, then the term `then` gives. */
function adaThen(then: (Person: any) => unknown): (txn: any) => any {
  return (txn: any) => {
    const Person = txn.crm.Person;
    return txn.uid("ada/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([
      (p: any) => p.symbols({ x: Person }).spec([(_: any) => _.x.name("Ada")])]).then([then(Person)]))]);
  };
}

/** An update writing x's age. */
function aged(age: bigint): (Person: any) => unknown {
  return (Person: any) => (w: any) => w.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: Person }).spec([(_: any) => _.x.age(age)])]));
}

/** A map binding to y the person named `name`. */
function met(name: string): (Person: any) => unknown {
  return (Person: any) => (w: any) => w.map((m: any) => m.symbols([
    (p: any) => p.symbols({ x: Person, y: Person }).spec([(_: any) => _.y.name(name)])]));
}

<a id="EXE-01"></a>
## EXE-01 · The transaction open, `txn`, its `exec` writing over one empty input frame, is `store.transaction`: the same frames and writes, committed with the scope, each frame carrying its input frame's id, then that input frame's completion marker; over a stream of input frames, each binding symbols by their references, each input frame's frames, then its marker

In [ ]:
{
  const [one, two] = [made(), made()];
  const given = one.transaction((txn: any = null) => {
    assert(txn instanceof Transactions.Transaction);
    const streamed = items(txn.exec(new Frame(), adaThen(aged(37n)), true));
    assert(equal(readPeople(one), [["Ada", 37n, ["Bob"]], ["Bob", null, []]])); // written within it
    return streamed;
  });
  assert(given.length === 2 && equal(given[1], map({ end: "0" }))); // its frame, then its completion marker
  const [done] = frames(given);
  assert(done.scope === "0" && done.remainder === null && done.errors.length === 0
    && equal(done.assigns, map({ x: { name: "Ada", age: 37n } })));
  const full = two.transaction(adaThen(aged(37n))); // its transaction's `exec`, collected
  assert(equal(names(one, [done]), names(two, full)) && equal(names(one, [done]), [map({ x: "Ada" })])
    && full[0].scope === "0");
  assert(equal(people(one), people(two)) && equal(people(one), [["Ada", 37n, ["Bob"]], ["Bob", null, []]]));
  one.transaction(() => { // it gave references, not visitors (CAP-315)
    assert(designated(one, done.get("x")) === one.singleton("crm.TheBook").contacts({ key: "ada" }));
  });
  const everyone = one.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([
    (p: any) => p.symbols({ x: t.crm.Person })]))]));
  const older = (t: any): any => t.uid("older/1").mutations([(mu: any) => mu.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: t.crm.Person }).spec([(_: any) => _.x.age(50n)])]))]); // x bound by each input frame
  const each = run(one, everyone.map((frame: any) => new Map([["x", frame.get("x")]])), older, true); // plain maps
  assert(equal(scopes(each), ["0", map({ end: "0" }), "1", map({ end: "1" })]));
  assert(equal(names(one, frames(each)), [map({ x: "Ada" }), map({ x: "Bob" })])
    && equal(people(one), [["Ada", 50n, ["Bob"]], ["Bob", 50n, []]]));
  const own = run(one, new Frame([["x", everyone[1].get("x")]], [], [], null, "bob"), older, true); // its id
  assert(own[0].scope === "bob" && equal(own[1], map({ end: "bob" })));
  raises(ValueError, () => one.transaction(older), "symbol 'x' of an update is bound by no term before it"); // none binds x
  raises(ValueError, () => one.transaction((txn: any = null) => {
    items(txn.exec(new Frame(), adaThen(aged(99n)), true));
    throw new ValueError("stop"); // the scope throws: what exec wrote is dropped with it
  }), "stop");
  assert(equal(people(one), [["Ada", 50n, ["Bob"]], ["Bob", 50n, []]]));
  const kept: any = one.transaction((txn: any = null) => txn);
  const ended = caught(() => items(kept.exec([new Frame()], adaThen(aged(98n)), true))); // used after its transaction
  assert(ended instanceof ValueError && ended.message.startsWith("a transaction's exec runs within it: this one has ended"));
}

<a id="EXE-02"></a>
## EXE-02 · Read-only: interpreted over what the store holds, writing nothing; a frame halts where a term would write, or where the store holds nothing to bind a symbol, given as bound so far, carrying its remainder, a position in the original expression; the same transaction, given it, completes it writing (the loopback), as a full run does, all committed with the scope

In [ ]:
{
  const [one, two] = [made(), made()];
  const older = adaThen(aged(40n));
  const [given, halted, looped] = one.transaction((txn: any = null) => {
    const streamed = items(txn.exec(new Frame(), older, false));
    const [stopped] = frames(streamed); // Ada is 36: the update would write
    assert(equal(readPeople(one), [["Ada", 36n, ["Bob"]], ["Bob", null, []]])); // nothing written
    const resumed = frames(items(txn.exec(stopped, stopped.remainder, true))); // the loopback: the same transaction
    assert(equal(readPeople(one), [["Ada", 40n, ["Bob"]], ["Bob", null, []]]));
    return [streamed, stopped, resumed];
  });
  assert(equal(given[1], map({ end: "0" })) && equal(names(one, [halted]), [map({ x: "Ada" })]) && halted.scope === "0");
  const remainder = halted.remainder;
  assert(equal(remainder.at, ["mutations", 0, "map", "then", 0, "update"])
    && remainder.subterm === "$.mutations[0].map.then[0].update");
  assert(String(remainder) === "<remainder $.mutations[0].map.then[0].update>");
  assert(remainder.equals(new Remainder(remainder.transaction, [...remainder.at])));
  assert(!remainder.equals(new Remainder(remainder.transaction, ["mutations", 0, "map"])) && !remainder.equals("elsewhere")
    && !remainder.equals(new Remainder(null, remainder.at)));
  const full = two.transaction(older);
  assert(equal(names(one, looped), names(two, full)) && equal(names(one, looped), [map({ x: "Ada" })]));
  assert(equal(people(one), people(two)) && equal(people(one), [["Ada", 40n, ["Bob"]], ["Bob", null, []]]));
  assert(frames(run(one, new Frame(), older, false))[0].remainder === null); // what holds is read, and given whole
  const cy = (t: any): any => t.uid("cy/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { c: t.crm.Person, book: t.crm.Book }).spec([(_: any) => _.book.contacts({ cy: (e: any) => e.contact(_.c) }),
    (_: any) => _.c.name("Cy")])]))]);
  const created = one.transaction((txn: any = null) => {
    const [creating] = frames(items(txn.exec(new Frame(), cy, false))); // an upsert that would create: as given to it
    assert(creating.size === 0 && equal(creating.remainder.at, ["mutations", 0, "upsert"]) && readPeople(one).length === 2);
    return frames(items(txn.exec(creating, creating.remainder, true)));
  });
  const fullCy = two.transaction(cy);
  assert(equal(names(one, created), names(two, fullCy)) && names(one, created)[0].get("c") === "Cy"
    && equal(people(one), people(two)));
  one.transaction((txn: any = null) => {
    const [unbound] = frames(items(txn.exec(new Frame(), adaThen(met("Dan")), false))); // nothing held binds y
    assert(equal(readNames(one, [unbound]), [map({ x: "Ada" })])
      && unbound.remainder.subterm === "$.mutations[0].map.then[0].map");
    assert(frames(items(txn.exec(unbound, unbound.remainder, true))).length === 0); // none, as a full run
  });
  assert(two.transaction(adaThen(met("Dan"))).length === 0);
  const note = (t: any): any => t.uid("note/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ n: t.crm.Note }).spec([(_: any) => _.n.text("x")])]))]);
  const retext = (t: any): any => t.uid("retext/1").mutations([(mu: any) => mu.update((u: any) => u.apply([(p: any) => p
    .symbols({ n: t.crm.Note }).spec([(_: any) => _.n.text("y")])]))]);
  const resumed = one.transaction((txn: any = null) => { // a counted object a frame binds, resumed within the transaction
    const [noted] = frames(items(txn.exec(new Frame(), note, true)));
    const [stopped] = frames(items(txn.exec(noted, retext, false))); // its text would be written
    assert(equal(stopped.remainder.at, ["mutations", 0, "update"]));
    const [again] = frames(items(txn.exec(stopped, stopped.remainder, true)));
    assert(again.errors.length === 0 && again.remainder === null);
    return again;
  });
  raises(LookupError, () => resumed.get("n"), "symbol 'n' of the frame: a counted 'crm.Note' has no reference"); // after it
}

<a id="EXE-03"></a>
## EXE-03 · A halt within a term: a map that binds some of its symbols, and nothing the store holds binds the rest, halts each frame as bound so far, its remainder that map and the terms after it; given again, the map binds the rest alone; a symbol given a reference that designates nothing halts alike

In [ ]:
{
  const one = made();
  const pair = (t: any): any => t.uid("pair/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols(
    { x: t.crm.Person, y: t.crm.Person }).spec([(_: any) => _.y.name("Cy")])]).then([
    (w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols({ x: t.crm.Person, y: t.crm.Person }).spec([
      (_: any) => _.x.knows([_.y])])]))]))]);
  const cy = (t: any): any => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols(
    { c: t.crm.Person, book: t.crm.Book }).spec([(_: any) => _.book.contacts({ cy: (e: any) => e.contact(_.c) }),
    (_: any) => _.c.name("Cy")])]))]);
  one.transaction((txn: any = null) => {
    const given = items(txn.exec(new Frame(), pair, false));
    const halted = frames(given); // x bound to each person, y to none: each halts within the map
    assert(equal(readNames(one, halted), [map({ x: "Ada" }), map({ x: "Bob" })])
      && equal(given[given.length - 1], map({ end: "0" })));
    assert(equal([...new Set(halted.map((frame: any) => frame.remainder.subterm))], ["$.mutations[0].map"]));
    items(txn.exec(new Frame(), cy, true)); // Cy, written meanwhile
    const looped = halted.flatMap((each: any) => frames(items(txn.exec(each, each.remainder, true))));
    assert(equal(readNames(one, looped), [map({ x: "Ada", y: "Cy" }), map({ x: "Bob", y: "Cy" })])); // y, the rest
  });
  assert(equal(people(one), [["Ada", 36n, ["Bob", "Cy"]], ["Bob", null, ["Cy"]], ["Cy", null, []]]));
  const bob = one.transaction(adaThen(met("Bob")))[0].get("y"); // Bob's reference
  one.transaction(() => {
    one.crm.Book(one.singleton("crm.TheBook"), (b: any) => b.contacts((a: any) => a.remove((e: any) => e.key("bob"))));
  });
  const named = (t: any): any => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ b: bob })]))]);
  one.transaction((txn: any = null) => {
    const [dangling] = frames(items(txn.exec(new Frame(), named, false))); // Bob deleted: designates nothing
    assert(dangling.size === 0 && equal(dangling.remainder.at, ["mutations", 0, "map"]));
    assert(frames(items(txn.exec(dangling, dangling.remainder, true))).length === 0); // none, as a full run
  });
}

<a id="EXE-04"></a>
## EXE-04 · Split-join through the loopback: read-only, a frame that halts within a branch of a partition is not given; the store that split keeps the join, the rest of that branch alone executed by the loopback, writing, within the same transaction, and joins what every branch ends with, going on after the join; as a full run

In [ ]:
/** Each person x, then a partition: its first branch makes x 36, its second binds whom x knows; then the book. */
function split(Person: any, Book: any): (txn: any) => any {
  const arm = (then: unknown) => (e: any): any => e.branch((b: any) => b.then([then]));
  return (txn: any) => txn.uid("split/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([
    (p: any) => p.symbols({ x: Person })]).then([(w: any) => w.partition((q: any) => q.branches([
    arm((w2: any) => w2.update((u: any) => u.apply([(p: any) => p.symbols({ x: Person }).spec([(_: any) => _.x.age(36n)])]))),
    arm((w2: any) => w2.map((m2: any) => m2.symbols([(p: any) => p.symbols({ x: Person, k: Person }).spec([
      (_: any) => _.x.knows([_.k])])]))),
  ]).then([(w3: any) => w3.map((m3: any) => m3.symbols([(p: any) => p.symbols({ bk: Book })]))]))]))]);
}
const splitting = (t: any): any => split(t.crm.Person, t.crm.Book)(t);
{
  const [one, two] = [made(), made()];
  const given = one.transaction((txn: any = null) => {
    const streamed = items(txn.exec(new Frame(), splitting, false));
    assert(equal(readPeople(one), [["Ada", 36n, ["Bob"]], ["Bob", 36n, []]])); // the loopback's write, in this transaction
    return streamed;
  });
  const joined = frames(given); // Ada holds the first branch; Bob halted in it, his rest executed by the loopback, and joined
  assert(joined.every((frame: any) => frame.remainder === null && frame.scope === "0")
    && equal(given[given.length - 1], map({ end: "0" })));
  const full = two.transaction(splitting);
  const shown = (store: any, frames_: any[]): string[] => names(store, frames_.map((frame) =>
    new Map([...frame].filter(([symbol]) => symbol !== "bk")))).map((each: Map<string, string>) =>
    JSON.stringify([...each].sort())).sort();
  assert(equal(shown(one, joined), shown(two, full)) && equal(shown(one, joined), [
    JSON.stringify([["k", "Bob"], ["x", "Ada"]]), JSON.stringify([["x", "Ada"]]), JSON.stringify([["x", "Bob"]])]));
  assert(joined.every((frame: any) => frame.has("bk"))); // the terms after the join, gone on with
  assert(equal(people(one), people(two)) && equal(people(one), [["Ada", 36n, ["Bob"]], ["Bob", 36n, []]])); // committed once
}

<a id="EXE-05"></a>
## EXE-05 · A branch's completion marker: a frame sent below with the rest of a branch carries the branch's id, its scope; executed alone, the rest of the branch and not the terms after the partition, its frames carry that id and end with its marker

In [ ]:
{
  const one = made();
  const expression = one.Mutations.OfTransaction(split(one.registered("crm.Person"), one.registered("crm.Book")));
  const rest = new Remainder(expression, ["mutations", 0, "map", "then", 0, "partition", "branches", 0, "then", 0, "update"]);
  const bob = one.transaction(adaThen(met("Bob")))[0].get("y");
  const given = run(one, new Frame([["x", bob]], [], [], null, "0/0"), rest, true);
  assert(equal(scopes(given), ["0/0", map({ end: "0/0" })]));
  assert(equal([...given[0].keys()], ["x"]) && equal(people(one), [["Ada", 36n, ["Bob"]], ["Bob", 36n, []]])); // no bk
}

<a id="EXE-06"></a>
## EXE-06 · Errors annotated in the stream: a frame a term fails to process is given annotated, the others too, nothing raised by `exec`, whether writing or read-only, or in the rest of a branch the loopback executes; the scope then commits nothing, raising the first error annotated, as `store.transaction` does; an input frame whose reference designates nothing, or that binds a counted object of another transaction, is given back annotated, and one given annotated as it is, executed no further

In [ ]:
{
  const one = made();
  const Person = one.registered("crm.Person");
  const plain = one.transaction(() => Plain.ToPlain(one).Transaction(one.Mutations.OfTransaction((t: any) => t
    .uid("knows/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p.symbols({ x: Person })]).then([
      (w: any) => w.partition((q: any) => q.branches([(e: any) => e.branch((b: any) => b.then([
        (w2: any) => w2.update((u: any) => u.apply([(p: any) => p.symbols({ x: Person }).spec([
          (_: any) => _.x.age(1n)])]).then([
          (w3: any) => w3.update((u3: any) => u3.apply([(p: any) => p.symbols({ x: Person }).spec([
            (_: any) => _.x.knows([one.singleton("crm.TheBook").contacts({ key: "bob" })])])]))]))]))]))]))]))));
  const knowing = Plain.FromPlain(one).Transaction(plain); // a spec naming Bob by his path, built while he is held
  const bob = one.transaction(adaThen(met("Bob")))[0].get("y");
  one.transaction(() => {
    one.crm.Book(one.singleton("crm.TheBook"), (b: any) => b.contacts((a: any) => a.remove((e: any) => e.key("bob"))));
  });
  const seen: any[] = [];
  const first = caught(() => one.transaction((txn: any = null) => {
    seen.push(...items(txn.exec(new Frame(), knowing, true))); // nothing thrown here: the frame is annotated
  }));
  assert(first instanceof LookupError && first.message === "no object of the store has the path a spec refers to");
  const [failed] = frames(seen); // Ada, the one person left: Bob's path names nothing
  assert(equal(names(one, [failed]), [map({ x: "Ada" })]) && equal(seen[1], map({ end: "0" })));
  assert(equal(failed.errors.map((e: any) => [e.subterm, e.kind, e.message]), [[
    "$.mutations[0].map.then[0].partition.branches[0].then[0].update.then[0].update", "LookupError",
    "no object of the store has the path a spec refers to"]]));
  assert(equal(people(one), [["Ada", 36n, []]])); // uncommitted, whole: Ada's age not written
  raises(LookupError, () => run(one, new Frame(), knowing, false), // read-only, in the rest of the branch the loopback runs
    "no object of the store has the path a spec refers to");
  raises(LookupError, () => one.transaction(knowing), "no object of the store has the path a spec refers to");
  const rest = new Remainder(knowing, ["mutations", 0, "map", "then", 0, "partition", "branches", 0, "then", 0, "update",
    "then", 0, "update"]);
  const ada = one.transaction(adaThen(aged(36n)))[0];
  let [early, dangling, again]: any[] = [];
  raises(LookupError, () => one.transaction((txn: any = null) => { // the first annotated
    [early] = frames(items(txn.exec(ada, rest, false))); // read-only, a term that fails before it would write
    [dangling] = frames(items(txn.exec(new Map([["x", bob]]), rest, true)));
    [again] = frames(items(txn.exec(early, rest, true))); // annotated: given back as it is
  }), "no object of the store has the path a spec refers to");
  assert(early.errors[0].kind === "LookupError" && early.remainder === null);
  assert(equal(dangling.errors.map((e: any) => [e.subterm, e.kind, e.message]), [[rest.subterm, "LookupError",
    "symbol 'x' of the frame: no object of the store has its reference's path"]]) && dangling.size === 0);
  assert(again.errors.every((e: any, i: number) => e.equals(early.errors[i])));
  const noted = one.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([
    (p: any) => p.symbols({ n: t.crm.Note }).spec([(_: any) => _.n.text("x")])]))]))[0];
  let refused: any = null;
  raises(LookupError, () => one.transaction((txn: any = null) => {
    [refused] = frames(items(txn.exec(noted, rest, true))); // its counted object, another transaction's
  }));
  assert(refused.errors[0].message.startsWith("symbol 'n' of the frame: a counted 'crm.Note' has no reference"));
}

<a id="EXE-07"></a>
## EXE-07 · An input stream consumed as it arrives: a generator's frames, each executed as it is taken, its frames given before the next is taken; a stream left before its end leaves what it wrote to the transaction; an error of no one frame is raised, carrying the frames given

In [ ]:
{
  const one = made();
  const everyone = one.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([
    (p: any) => p.symbols({ x: t.crm.Person })]))]));
  const older = (t: any): any => t.uid("older/2").mutations([(mu: any) => mu.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: t.crm.Person }).spec([(_: any) => _.x.age(60n)])]))]);
  const log: [string, unknown][] = [];
  function* arriving(): Generator<Map<string, unknown>> {
    for (const frame of everyone) {
      log.push(["taken", frame.scope]);
      yield new Frame([["x", frame.get("x")]]);
    }
  }
  one.transaction((txn: any = null) => {
    txn.exec(arriving(), older, true).subscribe((item: any) => {
      log.push(item instanceof Frame ? ["given", item.scope] : ["marker", item.get("end")]);
    });
  });
  assert(equal(log, [["taken", "0"], ["given", "0"], ["marker", "0"], ["taken", "0"], ["given", "1"], ["marker", "1"]]));
  assert(equal(people(one), [["Ada", 60n, ["Bob"]], ["Bob", 60n, []]]));
  const seventy = (t: any): any => t.uid("older/3").mutations([(mu: any) => mu.update((u: any) => u.apply([
    (p: any) => p.symbols({ x: t.crm.Person }).spec([(_: any) => _.x.age(70n)])]))]);
  one.transaction((txn: any = null) => {
    let firsts = 0;
    let started: any = null;
    const subscription = txn.exec(arriving(), seventy, true).subscribe({
      start: (given: any) => { started = given; }, // its subscription, before any item, as the proposal gives it
      next: () => { firsts += 1; started.unsubscribe(); }, // left after Ada's: what it wrote is the transaction's
      complete: () => { throw new Error("AssertionError: not completed"); },
    });
    assert(started === subscription && firsts === 1 && subscription.closed);
    const quiet = txn.exec(new Frame(), adaThen(aged(70n)), true);
    assert(quiet["@@observable"]() === quiet && quiet.subscribe().closed); // an observable, to RxJS and the proposal
    let completed = false;
    quiet.subscribe({ complete: () => { completed = true; } });
    assert(completed);
  });
  assert(equal(people(one), [["Ada", 70n, ["Bob"]], ["Bob", 60n, []]]));
  const thrown = caught(() => one.transaction((txn: any = null) => {
    let failure: unknown = null;
    txn.exec([new Frame([["x", everyone[0].get("x")]]), new Frame()], older, true).subscribe({ // the second binds no x
      error: (error: unknown) => { failure = error; } });
    txn.exec([new Frame()], older, true).subscribe({}); // its error given to none
    throw failure;
  }));
  assert(thrown instanceof ValueError && thrown.message === "symbol 'x' of an update is bound by no term before it");
  assert(equal(names(one, thrown.frames), [map({ x: "Ada" })])); // the frames given before it
  assert(equal(people(one), [["Ada", 70n, ["Bob"]], ["Bob", 60n, []]])); // the scope threw: nothing committed
}